In [ ]:
import os
import shutil
import subprocess
from huggingface_hub import login, HfApi

# =================【請填入您的資訊】=================
HF_TOKEN1 = "hf_xoNzunVEoJ"
HF_TOKEN2 = "FbFvjJPdCHaLeHxFtnMuJJxy"
FULL_TOKEN = f"{HF_TOKEN1}{HF_TOKEN2}"

HF_REPO_NAME = "ksamedandy/qwen36-heretic-progress"  
MODEL_ID = "unsloth/Qwen3.6-35B-A3B"
# ===================================================

login(token=FULL_TOKEN)
api = HfApi()

CHECKPOINT_DIR = "./heretic_study_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 🛠️ 【神級補丁】：即時修改 Binder 容器內 Heretic 的 model.py 原始碼！
# 我們強行把 `if get_model_class... == AutoModelForImageTextToText` 改成 `if False:`
# 這會徹底閹割掉多模態誤判，強迫 Heretic 把 Qwen 3.6 當作最純粹的文字模型來載入！
try:
    target_file = "/srv/conda/envs/notebook/lib/python3.10/site-packages/heretic/model.py"
    if os.path.exists(target_file):
        with open(target_file, "r") as f:
            code = f.read()
        code = code.replace(
            "if get_model_class(settings.model) == AutoModelForImageTextToText:",
            "if False:"
        )
        with open(target_file, "w") as f:
            f.write(code)
        print("✅ Heretic 原始碼多模態 Bug 已成功修復補丁！")
except Exception as e:
    print(f"⚠️ 補丁套用提示: {e}")

def backup_progress_to_cloud():
    print("\n📦 [自動同步] 正在將參數歷史上傳至 Hugging Face...")
    try:
        if os.path.exists(CHECKPOINT_DIR) and len(os.listdir(CHECKPOINT_DIR)) > 0:
            api.upload_folder(folder_path=CHECKPOINT_DIR, path_in_repo="study_checkpoints", repo_id=HF_REPO_NAME)
            print("🚀 [同步成功]\n")
    except Exception as e:
        print(f"❌ 同步失敗: {e}\n")

total_target_trials = 400 
current_completed = 0
batch_size = 5

print(f"🔥 [Binder 終極直擊] 開始在遠端大記憶體環境載入 35B 巨獸...")

while current_completed < total_target_trials:
    remaining = total_target_trials - current_completed
    run_trials = min(batch_size, remaining)
    target_trials_count = current_completed + run_trials
    
    print(f"\n🔄 正在執行第 {current_completed + 1} 至 {target_trials_count} 次參數試驗...")
    
    cmd = [
        "heretic",
        "--model", MODEL_ID,
        "--quantization", "BNB_4BIT",
        "--device-map", "auto",
        "--n-trials", str(target_trials_count),
        "--study-checkpoint-dir", CHECKPOINT_DIR,
        "--batch-size", "1",
        "--max-batch-size", "1",
        "--print-responses"
    ]
    
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    has_error = False
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            line = output.strip()
            print(line)
            if "Killed" in line or "Out of memory" in line or "error" in line.lower():
                has_error = True
                
    rc = process.poll()
    if rc != 0 or has_error:
        print(f"\n🏁 執行中斷 (Exit Code: {rc})。")
        break
        
    current_completed = target_trials_count
    backup_progress_to_cloud()

print("\n🏁 任務執行完畢。")
